In [ ]:
#[PHASE DIAGRAM

DELTA = 0.2
ALFA = -1.0
BETA = 1.0
GAMMA = 0.3
OMEGA = 1.2

def duffing(t, z):
    x, v = z
    dxdt = v
    dvdt = -DELTA*v - ALFA*x - BETA*x**3 + GAMMA*np.cos(OMEGA*t)
    return [dxdt, dvdt]

t_final = 1000  
t_eval = np.linspace(0, t_final, 100000)

x0 = 0.1
v0 = 0.0
z0 = [x0, v0]

sol = solve_ivp(duffing, [0, t_final], z0, t_eval=t_eval, rtol=1e-8, atol=1e-8)

x_val = sol.y[0]
v_val = sol.y[1]

cut = int(0.4 * len(x_val))

plt.figure(figsize=(8, 6))
plt.plot(x_val[cut:], v_val[cut:], color=(0.7, 0.2, 0.2), linewidth=0.3)
plt.xlabel("$x$")
plt.ylabel(r"$v = \frac{dx}{dt}$")
plt.title("Przestrzeń fazowa oscylatora Duffinga")
plt.grid(ls='--')
plt.show()

In [ ]:
def plot_phase_space(model, sample_params, t_eval):
    model.eval()
    with torch.no_grad():
        
        x_pred = model(sample_params).numpy().flatten()
    
    dt = t_eval[1] - t_eval[0]
    
    v_pred = np.gradient(x_pred, dt)
    
    plt.figure(figsize=(8, 6))
    plt.plot(x_pred, v_pred, label="Predykcja sieci", color='blue')
    plt.title("Przestrzeń fazowa oscylatora Duffinga")
    plt.xlabel(r"$ x(t)$")
    plt.ylabel(r"$ v(t) = \frac{dx}{dt}$")
    plt.grid(True)
    plt.legend()
    plt.show()

sample_idx = 0
sample_x, sample_y = test_dataset[sample_idx]
plot_phase_space(model, sample_x.unsqueeze(0), t_eval)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_attractor_with_vector_field(model, sample_params, t_eval):
    model.eval()
    
    params = sample_params.numpy().flatten()
    alpha, beta, delta, gamma, omega, _, _ = params

    with torch.no_grad():
        x_pred = model(sample_params).numpy().flatten()
    
    dt = t_eval[1] - t_eval[0]
    v_pred = np.gradient(x_pred, dt)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    x_min, x_max = x_pred.min() - 0.5, x_pred.max() + 0.5
    v_min, v_max = v_pred.min() - 0.5, v_pred.max() + 0.5
    
    X_grid, V_grid = np.meshgrid(np.linspace(x_min, x_max, 20), 
                                 np.linspace(v_min, v_max, 20))
    
    t_snapshot = 0 
    U = V_grid
    V = gamma * np.cos(omega * t_snapshot) - delta * V_grid - alpha * X_grid - beta * (X_grid**3)
    
    ax.quiver(X_grid, V_grid, U, V, color='blue', alpha=0.3, angles='xy', scale_units='xy')
    
    ax.scatter(x_pred, v_pred, s=0.5, color='midnightblue', alpha=0.6, label='Punkty trajektorii')
    
    ax.set_title("Przestrzeń fazowa oscylatora Duffinga z polem wektorowym")
    ax.set_xlabel("Położenie x(t)")
    ax.set_ylabel("Prędkość v(t) = dx/dt")
    ax.grid(False)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.show()


plot_attractor_with_vector_field(model, sample_x.unsqueeze(0), t_eval)